In [12]:
# ==========================================================
# SCANNER DE OPORTUNIDADES - LAY AWAY (PRODUÇÃO)
# ==========================================================
import pandas as pd
import numpy as np
import joblib
import requests
import io
import urllib.request
import os
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# --- 1. CONFIGURAÇÕES DE OPERAÇÃO ---
TOKEN = "b9f385cc07be27e7b04fe3a68c15120dd633d109"
ODD_MAX_LAY = 3.50
FILTRO_EDGE = 0.00
SPREAD_MAX = 0.50

# Configuração de data
intervalo_dias = False
dia_unico   = "2026-04-19"
data_inicio = "2026-04-01"
data_fim    = "2026-04-30"

# --- 2. CARREGAMENTO DO MODELO VIA GITHUB ---
url_modelo = 'https://github.com/tuedoidoe/LayAway5/raw/refs/heads/main/Modelo_LayAway_6.pkl'
caminho_local = 'Modelo_LayAway_6.pkl'

try:
    if not os.path.exists(caminho_local):
        print(f"Buscando cérebro do robô no GitHub...")
        urllib.request.urlretrieve(url_modelo, caminho_local)

    dados_modelo = joblib.load(caminho_local)
    model = dados_modelo['modelo']
    taxas_ligas = dados_modelo['liga_rates']
    media_global_treino = dados_modelo['media_global']
    features_modelo = dados_modelo.get('features', ["Market_Asymmetry", "Draw_Density", "Away_Odd_Trend", "Volatility_Risk", "LIGA_RATE"])

    print("✅ Inteligência Artificial carregada com sucesso!")
except Exception as e:
    print(f"❌ Erro crítico ao carregar modelo: {e}")

# --- 3. FUNÇÕES DE ORGANIZAÇÃO E CÁLCULO ---
def drop_reset_index(df):
    """Limpa NaNs e reseta o índice começando em 1"""
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

def safe_prob(column):
    return (1 / pd.to_numeric(column, errors='coerce').replace(0, np.nan)).fillna(0)

def baixar_jogos_do_dia(data):
    headers = {"Authorization": f"Token {TOKEN}"}
    url = f"https://apicomunidade.futpythontrader.com/api/dados/jogos-do-dia/betfair/{data}/download/"
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return pd.read_csv(io.BytesIO(response.content))
        return pd.DataFrame()
    except:
        return pd.DataFrame()

# --- 4. COLETA DOS JOGOS ---
lista_dias = pd.date_range(data_inicio, data_fim).strftime("%Y-%m-%d") if intervalo_dias else [dia_unico]
dfs = []

for dia in lista_dias:
    df_temp = baixar_jogos_do_dia(dia)
    if not df_temp.empty:
        df_temp["Date"] = dia
        dfs.append(df_temp)

if not dfs:
    print(f"🚨 Nenhum jogo encontrado para o período informado.")
else:
    # Consolidação inicial
    Jogos_do_Dia = pd.concat(dfs, ignore_index=True)
    Jogos_do_Dia = drop_reset_index(Jogos_do_Dia)
    print(f"🌍 Total de jogos carregados: {len(Jogos_do_Dia)}")

    # --- 5. PROCESSAMENTO E FILTROS ---
    cols_odds = ['Odd_A_Back', 'Odd_A_Lay', 'Odd_H_Back', 'Odd_H_Lay', 'Odd_Over25_FT_Back', 'Odd_CS_1x0_Lay', 'Odd_CS_2x1_Lay', 'Odd_CS_0x0_Lay', 'Odd_CS_1x1_Lay']
    for col in cols_odds:
        if col in Jogos_do_Dia.columns:
            Jogos_do_Dia[col] = pd.to_numeric(Jogos_do_Dia[col], errors='coerce')

    df_filt = Jogos_do_Dia[
        (Jogos_do_Dia['Odd_A_Lay'] <= ODD_MAX_LAY) &
        (Jogos_do_Dia['Odd_H_Back'] < Jogos_do_Dia['Odd_A_Back']) &
        (abs(Jogos_do_Dia['Odd_A_Back'] - Jogos_do_Dia['Odd_A_Lay']) <= SPREAD_MAX)
    ].copy()

    if df_filt.empty:
        print("⚠️ Nenhum jogo passou pelos filtros operacionais hoje.")
    else:
        # Engenharia de Atributos (Tropa de Elite)
        df_filt['Prob_1x2_A'] = safe_prob(df_filt['Odd_A_Back'])
        df_filt['Prob_CS_Resistance'] = safe_prob(df_filt['Odd_CS_1x0_Lay']) + safe_prob(df_filt['Odd_CS_2x1_Lay'])
        df_filt['Market_Asymmetry'] = (df_filt['Prob_CS_Resistance'] - df_filt['Prob_1x2_A'])
        df_filt['Draw_Density'] = safe_prob(df_filt['Odd_CS_0x0_Lay']) + safe_prob(df_filt['Odd_CS_1x1_Lay'])
        df_filt['Volatility_Risk'] = np.clip((df_filt['Odd_Over25_FT_Back'] / df_filt['Odd_A_Back'].replace(0, np.nan)), 0, 50)
        df_filt['Away_Odd_Trend'] = 0.0

        # Mapeando a Memória das Ligas
        df_filt['LIGA_RATE'] = df_filt['League'].map(taxas_ligas).fillna(media_global_treino)

        # --- 6. PREDIÇÃO E EDGE ---
        X_today = df_filt[features_modelo]
        df_filt['Prob_IA'] = model.predict_proba(X_today)[:, 1]
        df_filt['Prob_Mercado'] = 1 - (1 / df_filt['Odd_A_Lay'])
        df_filt['Edge'] = df_filt['Prob_IA'] - df_filt['Prob_Mercado']

        # --- 7. EXIBIÇÃO DOS SINAIS ---
        df_sinais = df_filt[df_filt['Edge'] >= FILTRO_EDGE].copy()

        if df_sinais.empty:
            print(f"📊 {len(df_filt)} jogos analisados. Nenhum atingiu o Edge de {FILTRO_EDGE:.2%}.")
        else:
            print("\n" + "="*80)
            print(f"🎯 SINAIS ENCONTRADOS")
            print("="*80)

            exibicao = df_sinais[["Date", "Time", "League", "Home", "Away", "Odd_A_Lay", "Edge"]]

            # Organização final com a função solicitada
            exibicao = drop_reset_index(exibicao)

            # Formatação para leitura humana
            exibicao['Edge'] = (exibicao['Edge'] * 100).round(2).astype(str) + '%'

            from IPython.display import display
            display(exibicao)

✅ Inteligência Artificial carregada com sucesso!
🌍 Total de jogos carregados: 131

🎯 SINAIS ENCONTRADOS


,Date,Time,League,Home,Away,Odd_A_Lay,Edge
1,2026-04-19,02:30:00,AUSTRALIA 1,Adelaide United,Macarthur FC,3.35,1.61%
2,2026-04-19,08:30:00,TURKEY 1,Kasimpasa,Alanyaspor,2.98,3.16%
3,2026-04-19,11:00:00,DENMARK 1,FC Nordsjaelland,Viborg,3.35,8.89%
4,2026-04-19,12:00:00,NORWAY 2,Odds BK,Stabaek,2.72,0.85%
5,2026-04-19,12:15:00,FRANCE 1,Nantes,Brest,3.35,3.64%
